In [ ]:
# Install the RDKit library using pip
!pip install rdkit==2026.3.1

In [ ]:
#To confirm the installation, let's import the library and check its version.
import rdkit
print(rdkit.__version__)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

In [ ]:
#structure of decane
from rdkit import Chem
decane = Chem.MolFromSmiles("CCCCCCCCCC")
decane

In [ ]:
#structure of nonane
from rdkit import Chem
nonane = Chem.MolFromSmiles("CCCCCCCCC")
nonane

In [ ]:
#Downolad Structure
from rdkit import Chem

octane = Chem.MolFromSmiles("CCCCCCCC")

Chem.MolToMolFile(octane, "octane.sdf")
Chem.MolToMolFile(octane, "octane.mol")

In [ ]:
#heptene
heptene = Chem.MolFromSmiles("CCCCCC=C")
heptene

In [ ]:
# 2,2-dimethylpropane (neopentane)
mol2 = Chem.MolFromSmiles("CC(C)(C)C")
mol2

In [ ]:
#Draw the structure of the following
2,2-dimethylheptane
3,3-dimethylheptane
2,3-dimethylheptane
2,4-dimethyloctane
cyclooctane
cyclodecane

In [ ]:
##(R)-2-hexanol
rhexanol=Chem.MolFromSmiles("C[C@@H](O)CCCC")
rhexanol

In [ ]:
##(S)-2-hexanol
shexanol=Chem.MolFromSmiles("C[C@H](O)CCCC")
shexanol

In [ ]:
from rdkit import Chem

from rdkit.Chem import Descriptors

In [ ]:
#calculate the molecular weight of pentane
#pentane
pentane_smiles = "CCCCCC"
pentane_mol = Chem.MolFromSmiles(pentane_smiles)
# Calculate Molecular Weight
pentane_mw = Descriptors.MolWt(pentane_mol)
print(f"Molecular Weight of Pentane: {pentane_mw:.2f} g/mol")

In [ ]:
#Calculate the Molecular weight of the following, make use of the above code by modifying it
#hexanol
#pentanol
#Butane
#Aspirin
#Decane

In [ ]:
# Install all required libraries
!pip install rdkit pubchempy requests -q
print("All libraries installed successfully!")

In [ ]:
# Import libraries for working with molecules
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem
from rdkit.Chem import SaltRemover

# Import library for PubChem
import pubchempy as pcp

# Other helpful libraries
import requests
import pandas as pd

print("All imports successful!")

In [ ]:
#Run the code to download the structure of Aspirin from Pubchem
import requests

name = "aspirin"

url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF"

response = requests.get(url)

with open("aspirin.sdf", "w") as file:
    file.write(response.text)

print("Aspirin downloaded!")

In [ ]:
#Dowload the structure of the following compounds from PubChem
#hexanol
#pentanol
#Butane
#Mangiferin
#Decane

In [ ]:
#Run the following code to perform the following tasks
#20 compounds → PubChem → SDF → Canonical SMILES → RDKit → Lipinski screening → results table → Excel file


import requests
import pandas as pd
import os

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski


# ==========================================
# 1. List of 20 compounds
# ==========================================

molecules = [

    "acetone",

    "ethanol",

    "methanol",

    "glucose",

    "fructose",

    "Decane",

    "Heptanol",

    "salicylic acid",

    "Mangiferin",

    "Rutin",

    "nicotinamide",

    "riboflavin",

    "Acetone",

    "aniline",

    "phenol",

    "toluene",

    "Decanol",

    "paroxetine",

    "testosterone",

    "progesterone"

]


# ==========================================
# 2. Create a folder for the SDF files
# ==========================================

folder = "molecules20"
os.makedirs(folder, exist_ok=True)


# ==========================================
# 3. Create an empty list for the results
# ==========================================

data = []


# ==========================================
# 4. Download and analyze each compound
# ==========================================

for number, name in enumerate(molecules, start=1):

    print(f"Processing {name}...")

    # PubChem URL for Canonical SMILES
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/CanonicalSMILES/JSON"

    response = requests.get(url)

    if response.status_code == 200:

        result = response.json()

        # Get Canonical SMILES
        smiles = result["PropertyTable"]["Properties"][0]["ConnectivitySMILES"]

        # Convert SMILES to an RDKit molecule
        molecule = Chem.MolFromSmiles(smiles)

        # ==========================================
        # 5. Download SDF file
        # ==========================================

        sdf_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF"

        sdf_response = requests.get(sdf_url)

        with open(f"{folder}/{name}.sdf", "w") as file:
            file.write(sdf_response.text)


        # ==========================================
        # 6. Calculate Lipinski properties
        # ==========================================

        molecular_weight = Descriptors.MolWt(molecule)

        logp = Descriptors.MolLogP(molecule)

        hbd = Lipinski.NumHDonors(molecule)

        hba = Lipinski.NumHAcceptors(molecule)


        # ==========================================
        # 7. Count Lipinski violations
        # ==========================================

        violations = 0

        if molecular_weight > 500:
            violations += 1

        if logp > 5:
            violations += 1

        if hbd > 5:
            violations += 1

        if hba > 10:
            violations += 1


        # ==========================================
        # 8. Determine Lipinski result
        # ==========================================

        if violations <= 1:
            rule = "Pass"
        else:
            rule = "Fail"


        # ==========================================
        # 9. Store results
        # ==========================================

        data.append([
            number,
            name,
            smiles,
            molecular_weight,
            logp,
            hbd,
            hba,
            violations,
            rule
        ])

    else:

        print(f"{name} could not be found on PubChem")


# ==========================================
# 10. Create results table
# ==========================================

table = pd.DataFrame(
    data,
    columns=[
        "S/N",
        "Compound",
        "Canonical SMILES",
        "Molecular Weight",
        "LogP",
        "HBD",
        "HBA",
        "Violations",
        "Lipinski Rule"
    ]
)


# ==========================================
# 11. Display the table
# ==========================================

display(table)